# Cell labels on the corpus baseline

The offline classifier of `docs/spec/CELL_FACTS.md` (Half 2: syntactic, multi-label, versioned; rules in
`labels.json`) run over every `exec` and `python_exec` cell of the corpus baseline: one model,
`qwen3.8-flash-next:vision`, 38 tasks x 3 repeats on build `9be83ae0`, corpus version 1.

**Label: "model told: stateless, no filesystem (defect)".** These runs were given the page-hosted `python_exec`
description, which says there is no filesystem and every call is stateless. Where the model reaches for JS, or probes
its environment, that is consistent with what it was told and is not evidence about the model. Reruns A and B are the
comparison.

The questions: (1) does every cell parse, and with which parser; (2) which kinds of step appear in which language, by
stratum; (3) for Python, which kinds ran inside the dialect, were asked, or never ran; (4) what the forms stratum's cells
are, as the input to forms step c (score the stratum by whether the invited form ran).

In [1]:
import sys
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "cell-labels"
sys.path.insert(0, str(HERE))
import baseline, classify

DATA = HERE / "data" / "cell-labels-baseline"
pd.set_option("display.width", 160, "display.max_rows", 80)
SOURCE = {"sweep": "corpus", "reader": "notebooks/cell-labels/baseline.py", "classifier": "notebooks/cell-labels/classify.py",
          "taxonomy": classify.TAXONOMY, "privacy": "labels only: no cell code"}
rows = pd.DataFrame(wmlnb.extract(DATA / "cells.json", lambda: baseline.read(wmlnb.find_sweep("corpus")), source=SOURCE))
assert (rows.taxonomy == classify.TAXONOMY).all(), "the extract was classified under another taxonomy: NB_REFRESH=1"
lab = rows.explode("labels").rename(columns={"labels": "label"}).reset_index(drop=True)
LANG = {"python": "Python", "js": "JS"}
pd.DataFrame(wmlnb.provenance(DATA / "cells.json"))

,extract,sha256,rows,extracted_at,commit,classifier,privacy,reader,sweep,taxonomy
0,notebooks/cell-labels/data/cell-labels-baselin...,c29fdbe190bc319c,715,2026-11-24T09:45:08+00:00,fd4ad46f09e2,notebooks/cell-labels/classify.py,labels only: no cell code,notebooks/cell-labels/baseline.py,corpus,1


## Every cell, and its parser

A JS cell is read by the dialect's own parser when it can be, and by acorn when the dialect refuses it; a cell neither
reads is `unparsed`. A Python cell is read by `ast`.

In [2]:
t = pd.crosstab(rows.lang.map(LANG), rows.parser.fillna("unparsed"), margins=True, margins_name="all")
t

parser,acorn,ast,dialect,unparsed,all
lang,,,,,
JS,108,0,231,5,344
Python,0,370,0,1,371
all,108,370,231,6,715


## Which kind of step goes to which language, by stratum

The share of cells carrying each label (a cell carries several, so a column does not sum to 100). `n` is the cells in
the column. `files` tasks have attached files, `inline` tasks a table framed whole into the message, `forms` tasks
invite a loop-like form.

In [3]:
n = rows.groupby(["lang", "stratum"]).size()
share = (lab.groupby(["lang", "stratum"]).label.value_counts() / n).unstack(["lang", "stratum"]).fillna(0)
share = share.rename(columns=LANG, level=0).sort_index(axis=1)
order = ["ingest", "parse", "list", "probe", "inspect", "select", "transform", "aggregate", "report", "define",
         "dom-read", "dom-act", "write", "shell", "plot", "other", "unparsed"]
out = (100 * share.reindex([l for l in order if l in share.index])).round(0).astype(int)
out.loc["n"] = n.rename(index=LANG, level=0).reindex(out.columns).values
out

lang         JS              Python             
stratum   files forms inline  files forms inline
label                                           
ingest       49    53     21     74    68     81
parse        34    40     37      2     7      5
list         16    17     11     24    26     19
probe        33    36     74      8     4      8
inspect      44    60     47     67    60     73
select       35    26     37     32    35     16
transform    60    64     63     33    39     59
aggregate    30    30     29     44    44     59
report       47    45     61     18    14     11
define       20    13      8      3     9      0
dom-read      5     4      5      0     0      0
write         1     0      0      0     0      0
other         1     2      0      1     5      0
unparsed      2     0      0      0     0      0
n           259    47     38    277    57     37

The same, counted per language over all strata: how many cells of each kind each language got, and the JS share of
that kind.

In [4]:
c = lab.groupby("label").lang.value_counts().unstack().fillna(0).astype(int).rename(columns=LANG)
c["JS share %"] = (100 * c["JS"] / (c["JS"] + c["Python"])).round(0).astype(int)
c.reindex([l for l in order if l in c.index])

lang,JS,Python,JS share %
label,,,
ingest,160,273,37
parse,120,11,92
list,53,89,37
probe,131,26,83
inspect,159,246,39
select,116,116,50
transform,210,136,61
aggregate,103,168,38
report,165,62,73


## Python: which kinds ran inside the dialect

A Python cell's outcome, from the sweep's extract: `inside` ran inside the read-only dialect, `asked` ran outside it
(the person was asked), `not run` was refused before any code ran (its `tables` argument failed). The not-run cells'
labels describe what the model wrote, not what ran. Shares are of the cells carrying the label.

In [5]:
py = lab[lab.lang == "python"]
assert rows[rows.lang == "python"].outcome.notna().all(), "a Python cell did not join to the sweep's extract"
o = pd.crosstab(py.label, py.outcome)
o = o.reindex([l for l in order if l in o.index])
pct = (100 * o.div(o.sum(axis=1), axis=0)).round(0).astype(int).add_suffix(" %")
pd.concat([o.sum(axis=1).rename("cells"), pct], axis=1)

,cells,asked %,inside %,not run %
label,,,,
ingest,273,45,2,53
parse,11,82,9,9
list,89,100,0,0
probe,26,58,0,42
inspect,246,44,0,55
select,116,70,0,30
transform,136,65,1,35
aggregate,168,62,1,38
report,62,71,0,29


## Flags

`loop` is `for`, `while` or a comprehension; `returns` is a cell that ends in a return or a bare expression (a `print`
or `console.log` is not one); `by-argument` is a `python_exec` call that passed `tables`, which earns `ingest` by itself.

In [6]:
fl = rows.explode("flags").dropna(subset=["flags"]).reset_index(drop=True)
f = (100 * fl.groupby("lang")["flags"].value_counts() / rows.groupby("lang").size()).unstack(0).fillna(0).round(0).astype(int)
f.rename(columns=LANG)

lang,JS,Python
flags,,
by-argument,0,39
import,0,75
loop,35,23
returns,57,29
try,5,8


## The forms stratum, task by task

The forms tasks invite `while`, recursion, a class or a generator. Per task and language: the cells, the share with a
loop, the share that define a function or class, and the share that earn only `other` (arithmetic with no data step,
the shape of a closed-form answer).

In [7]:
fm = rows[rows.stratum == "forms"].copy()
fm["loop"] = fm["flags"].map(lambda x: "loop" in x)
fm["define"] = fm.labels.map(lambda x: "define" in x)
fm["only other"] = fm.labels.map(lambda x: x == ["other"])
g = fm.groupby(["task", fm.lang.map(LANG)]).agg(cells=("step", "size"), loop=("loop", "mean"), define=("define", "mean"), only_other=("only other", "mean"))
g[["loop", "define", "only_other"]] = (100 * g[["loop", "define", "only_other"]]).round(0).astype(int)
g.unstack().fillna(0).astype(int)

cells        loop        define        only_other       
lang                          JS Python   JS Python     JS Python         JS Python
task                                                                               
forms-chunk-max                8     12   12     75      0      0          0      0
forms-cumulative-threshold     9      4   33      0     11      0          0      0
forms-double-weeks             0      5    0     20      0      0          0     60
forms-org-depth               13      9   38     44     23     33          8      0
forms-ship-backlog            10     11   30     27      0      0          0      0
forms-store-balance            7     16   29     44     29     12          0      0

## What it shows

- **Hand-parsing is a JS behaviour.** 120 of the 131 `parse` cells are JS (92%), a third or more of the JS cells in
  every stratum; Python reads the same files with `pd.read_csv` (`ingest`).
- **The JS cells probe.** 131 JS cells against 26 Python carry `probe`; 98 of those 131 earn it only by reading
  `ml.current` (a rule choice, below): the model looks for its files through the run's context, from JS.
- **Every Python `list` cell was asked** (89 of 89): `os.listdir` and its kin are outside the dialect. Few cells of any
  kind ran inside it, as the baseline's class A count (9 of 226 cells that ran) already said.
- **The forms stratum barely shows its forms.** `forms-double-weeks` has no JS cell and 3 of its 5 Python cells earn
  only `other`, the shape of a closed-form answer; `forms-cumulative-threshold` has 4 Python cells with no loop.
  This is the input forms step c will score from.

## Choices the data file makes

The spec's table gives each label's leading names; `labels.json` is the rule. Where it goes past the table:

- Reading `ml.current` is `probe` (the cell asks what its run holds), and reading `ml.current.files` or
  `ml.current.attachments` is also `list` (file metadata).
- `df.to_csv()` with no path returns text, so it is `report`, not `write`.
- A call in a chain that feeds a separator `split` (`txt.trim().split("\n")`) is part of the parse, like the `map`
  chained on it.
- An accumulation in a Python loop (`total += x`) is `aggregate`, as the spec says for JS's `for…of`.
- A `python_exec` call whose `tables` is an empty string passed no table, so it earns no `by-argument`.